# Ondate di calore marine nel Mediterraneo — tutorial MHEAT

## Contesto scientifico

Un'**Ondata di Calore Marina (MHW)** è un evento oceanico discreto, prolungato e anomalamente caldo. Formalmente, Hobday *et al.* (2016) definiscono una MHW come un periodo di almeno **cinque giorni consecutivi** durante il quale la temperatura superficiale del mare (SST) supera il 90° percentile variabile stagionalmente, calcolato su una climatologia di riferimento di 30 anni (tipicamente 1991–2020). Una volta rilevato, l'evento è classificato in cinque categorie di severità (I Moderata → V Super-Estrema) in base al rapporto tra anomalia di picco e anomalia di soglia. Poiché la definizione è relativa alla climatologia locale, la stessa temperatura assoluta può essere una MHW in una regione e perfettamente normale in un'altra.

Il **Mar Mediterraneo** è uno dei bacini oceanici che si riscaldano più velocemente: le temperature superficiali sono aumentate di circa 0,4 °C per decennio dagli anni '80, cioè circa tre volte la media globale, e il bacino ha sperimentato MHW estese a tutto il bacino nel 2003, 2015, 2017, 2018, 2022 e 2023. Questi eventi coincidono con mortalità massive di invertebrati bentonici (gorgonie, spugne, bivalvi), perdite nell'acquacoltura dei pesci e sbiancamenti su larga scala delle praterie di *Posidonia oceanica* — un habitat chiave del Mediterraneo. Quantificare la co-localizzazione degli eventi estremi con i settori vulnerabili è un'esigenza politica diretta ai sensi della Direttiva Quadro sulla Strategia per l'Ambiente Marino e della Strategia UE sulla Biodiversità 2030.

Questo notebook percorre il flusso di lavoro di MHEAT su un **cubo SST sintetico incluso** in modo da poter eseguire ogni cella **senza un account Copernicus** o accesso a Internet. Lo stesso codice, invariato, funziona su dati reali del Copernicus Marine Service una volta forniti i credenziali — vedere la sezione finale.

---

## Dipendenze

```bash
pip install xarray numpy pandas matplotlib marineHeatWaves netCDF4
# Opzionale, per recuperi in diretta dal CMS:
pip install copernicusmarine
```

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Make the MHEAT backend importable. Adds any of (repo/backend, /srv/app)
# depending on whether the notebook is run from a clone or inside the
# Docker image.
for candidate in (os.path.abspath('../backend'), '/srv/app'):
    if os.path.isdir(os.path.join(candidate, 'app')) and candidate not in sys.path:
        sys.path.insert(0, candidate)
print('Python', sys.version.split()[0])

## Passo 1 — Caricare un cubo SST del Mediterraneo

Usiamo il cubo sintetico fornito con MHEAT. È un campo SST giornaliero di 3 anni (2020–2022) a 0,25° che copre la maggior parte del Mediterraneo occidentale e centrale, con un ciclo stagionale realistico e una forte anomalia calda iniettata in luglio–agosto 2022 per imitare l'ondata di calore storica di quell'estate.

Il cubo ha forma ``(1096 giorni × 41 lat × 73 lon)``.

In [ ]:
from app.sst import build_synthetic_med_cube

ds = build_synthetic_med_cube(anomaly_year=2022)
print(ds)

## Passo 2 — Climatologia stagionale + soglia al 90° percentile

Selezioniamo il pixel più caldo per l'illustrazione, eseguiamo ``marineHeatWaves.detect()`` e tracciamo le tre curve di riferimento: la SST giornaliera, la media lisciata per giorno dell'anno (climatologia) e la soglia al 90° percentile. Le aree rosse ombreggiate sono le MHW che il rilevatore ha segnalato.

In [ ]:
from marineHeatWaves import detect as mhw_detect

da = ds['analysed_sst']

# Warmest pixel at the 2022 peak day.
peak_day = da.sel(time='2022-07-25').mean(['latitude', 'longitude']).item()
# Actually choose the hottest pixel on that date.
peak_slice = da.sel(time='2022-07-25')
iy, ix = np.unravel_index(int(np.argmax(peak_slice.values)), peak_slice.shape)
lat_val = float(da['latitude'][iy]); lon_val = float(da['longitude'][ix])
print(f'Hottest pixel at 2022-07-25: lat={lat_val:.2f}° lon={lon_val:.2f}°')

series = da.isel(latitude=iy, longitude=ix).values.astype('float64')
times = pd.to_datetime(da.time.values)
ordinals = np.array([t.toordinal() for t in times])

mhws, clim = mhw_detect(
    ordinals, series,
    climatologyPeriod=[times[0].year, times[-1].year - 1],
    pctile=90, windowHalfWidth=5,
    minDuration=5, joinAcrossGaps=True, maxGap=2,
)
print(f"Events detected on this pixel: {mhws['n_events']}")
for i in range(mhws['n_events']):
    s = pd.Timestamp.fromordinal(int(mhws['time_start'][i]))
    e = pd.Timestamp.fromordinal(int(mhws['time_end'][i]))
    print(f"  #{i+1}: {s.date()} → {e.date()}  duration={int(mhws['duration'][i])}d  i_max={mhws['intensity_max'][i]:.2f}°C")

In [ ]:
# Plot 1 — time series with threshold bands
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, series, label='SST', color='black', lw=0.8)
ax.plot(times, clim['seas'], label='Climatology (DOY mean)', color='tab:blue')
ax.plot(times, clim['thresh'], label='90th percentile', color='tab:orange', ls='--')
for i in range(mhws['n_events']):
    s = pd.Timestamp.fromordinal(int(mhws['time_start'][i]))
    e = pd.Timestamp.fromordinal(int(mhws['time_end'][i]))
    ax.axvspan(s, e, color='red', alpha=0.25)
ax.set_ylabel('SST (°C)')
ax.set_title(f'Hobday MHW detection — pixel ({lat_val:.2f}°N, {lon_val:.2f}°E)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Passo 3 — Rilevamento pixel per pixel + clustering con gli helper MHEAT

Il backend incapsula ``marineHeatWaves.detect`` in ``detect_cube`` (pixel per pixel) e ``cluster_events`` (unisce i vicini spazio-temporali). Insieme trasformano migliaia di piccoli eventi per pixel in una manciata di poligoni di regione contigui — l'unità di analisi giusta per gli studi di impatto.

In [ ]:
from app.mhw import detect_cube, cluster_events

events = detect_cube(da, clim_period=(2020, 2021), max_pixels=400)
print(f'Per-pixel events detected: {len(events)}')

clusters = cluster_events(events)
print(f'After clustering: {len(clusters)}')
for c in clusters[:5]:
    print(f"  {c.event_id}  {c.category_name}  {c.date_start} → {c.date_end}  n_pixels={c.n_pixels}")

## Passo 4 — Mappa spaziale della densità di eventi + istogramma di intensità

In [ ]:
# Plot 2 — spatial map of per-pixel event counts, derived from the detected events
grid = np.zeros((da.sizes['latitude'], da.sizes['longitude']))
lat_vals = da['latitude'].values
lon_vals = da['longitude'].values
for e in events:
    iy = int(np.argmin(np.abs(lat_vals - e.centroid_lat)))
    ix = int(np.argmin(np.abs(lon_vals - e.centroid_lon)))
    grid[iy, ix] += 1

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.pcolormesh(lon_vals, lat_vals, grid, cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax, label='# events at pixel')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Per-pixel MHW event count — 2020–2022 synthetic Mediterranean')
# Overlay cluster centroids as black stars
for c in clusters:
    ax.plot(c.centroid_lon, c.centroid_lat, marker='*', color='black', ms=12, mew=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 3 — intensity histogram by Hobday category
fig, ax = plt.subplots(figsize=(8, 4))
intens = [e.intensity_max for e in events]
cats = [e.category for e in events]
cat_colors = {1:'#ffd166', 2:'#ff9f1c', 3:'#e63946', 4:'#9d0208', 5:'#370617'}
cat_labels = {1:'I Moderate', 2:'II Strong', 3:'III Severe', 4:'IV Extreme', 5:'V Super-Extreme'}
bins = np.linspace(0, max(intens) + 0.5, 20)
for c in sorted(set(cats)):
    sub = [i for i, cc in zip(intens, cats) if cc == c]
    ax.hist(sub, bins=bins, alpha=0.7, label=cat_labels[c], color=cat_colors[c])
ax.set_xlabel('Peak intensity (°C above threshold)')
ax.set_ylabel('# events')
ax.set_title('Distribution of per-pixel MHW peak intensities')
ax.legend()
plt.tight_layout(); plt.show()

## Passo 5 — Giunzione con gli strati settoriali

Il backend spedisce un piccolo pacchetto di siti di acquacoltura mediterranei, AMP marine Natura 2000 e poligoni di prateria di posidonia sotto ``backend/app/fixtures/overlays/``. ``attach_impact_properties`` riporta, per evento, quanti siti di acquacoltura sono colpiti e quanti km² di habitat AMP / posidonia il poligono dell'evento copre.

In [ ]:
import json
from pathlib import Path
from app.impact import attach_impact_properties
from app.mhw import events_to_geojson

# Find the overlay fixtures dir — either in the cloned repo or inside the container.
candidates = [Path('../backend/app/fixtures/overlays'), Path('/srv/app/app/fixtures/overlays')]
fixtures_dir = next((p for p in candidates if p.is_dir()), candidates[0])

overlays = {k: json.loads((fixtures_dir / f'{k}.json').read_text()) for k in ('aquaculture','mpa','seagrass')}

geojson = events_to_geojson(clusters)
attach_impact_properties(geojson, clusters, overlays)
for f in geojson['features'][:3]:
    p = f['properties']
    imp = p.get('impact', {})
    print(f"{f['id']}  {p['category_name']}  aquaculture={imp.get('n_aquaculture_sites')}  mpa_km2={imp.get('mpa_area_km2')}  seagrass_km2={imp.get('seagrass_area_km2')}")

## Come eseguire questo notebook su EDITO Datalab

1. Nel EDITO Datalab, avviare il servizio **JupyterLab** (qualsiasi immagine Python 3.11).
2. Aprire un terminale ed eseguire `git clone https://github.com/<your-org>/mheat.git`, poi `cd mheat/tutorials` e aprire questo notebook.
3. `pip install -r ../backend/requirements.txt` dall'ambiente del notebook (o integrare le dipendenze nella propria immagine).
4. Per eseguire contro i **veri** dati Copernicus Marine, impostare `COPERNICUSMARINE_SERVICE_USERNAME` e `COPERNICUSMARINE_SERVICE_PASSWORD` nell'ambiente del notebook e sostituire il caricamento sintetico con:

```python
import copernicusmarine
out = copernicusmarine.subset(
    dataset_id='SST_MED_SST_L4_NRT_OBSERVATIONS_010_004',
    minimum_longitude=-6, maximum_longitude=36.5,
    minimum_latitude=30, maximum_latitude=46,
    start_datetime='2022-05-01T00:00:00', end_datetime='2022-09-30T23:59:59',
    variables=['analysed_sst'],
)
ds = xr.open_dataset(out)
```

5. Il resto del notebook viene eseguito senza modifiche. Il servizio MHEAT stesso (``docker compose up --build``) è anche ridistribuibile su un'istanza EDITO **Onyxia** — puntare `FRONTEND_DIR` al bundle Vite compilato ed esporre la porta 8000.

## Citazioni

* Hobday, A. J., Alexander, L. V., Perkins, S. E., Smale, D. A., *et al.* (2016). *A hierarchical approach to defining marine heatwaves.* **Progress in Oceanography**, 141, 227–238.
* Hobday, A. J., Oliver, E. C. J., *et al.* (2018). *Categorizing and naming marine heatwaves.* **Oceanography**, 31(2), 162–173.
* Oliver, E. C. J. (2019). `marineHeatWaves` — pacchetto Python per l'identificazione delle ondate di calore marine. https://github.com/ecjoliver/marineHeatWaves
* Copernicus Marine Service — https://marine.copernicus.eu